In [1]:
import os

from dotenv import load_dotenv

print(os.getenv("TAVILY_API_KEY"))

None


In [10]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

from dotenv import load_dotenv

load_dotenv()

search_tool = TavilySearch(
    max_results=5,
    topic="general",  # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)



# search_tool.invoke("中国有多少人")

In [14]:

import base64
import base64
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


@tool
def web_search(query : str):
    """进行web搜索"""
    return search_tool.invoke(input=query)


# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")


# Agent的回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

qwen = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.7,
)

agent = create_agent(
    model=qwen,
    tools=[search_tool],
    system_prompt="你是一个智能助手,你使用工具来解决用户问题",
    response_format=AnswerInfo,
)

# 阻塞式调用
response = agent.invoke(
    input={"messages": [HumanMessage("百度公司厉害吗")]},
)

for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

百度公司厉害吗
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_72032ad5789b4adb9cf057)
 Call ID: call_72032ad5789b4adb9cf057
  Args:
    query: 百度公司实力如何
================================= Tool Message =================================
Name: tavily_search

{"query": "百度公司实力如何", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.sohu.com/a/679257948_121687424", "title": "百度公司怎么样？ - 搜狐", "content": "百度作为中国最大的互联网搜索引擎公司之一，其搜索引擎技术是其最大的优势。百度公司的搜索引擎技术已被不断地升级和完善，具有很高的精度和稳定性。其核心", "score": 0.9994253, "raw_content": null}, {"url": "https://m.cyzone.cn/article/691206", "title": "百度：到2030年可能成为中国市值最高的公司", "content": "百度目前是中国市值排名第34位的公司。我相信，到2030年，百度将成为世界排名第一的公司。 百度股价从历史高点下跌了约55.5%，我", "score": 0.9989759, "raw_content": null}, {"url": "https://baike.baidu.com/item/%E7%99%BE%E5%BA%A6/6699", "title": "公司简介 - 百度百科", "content